##### Working - After directly going to company profile, extracts: "NAICS Code", "Industry", "Description", "Website URL", "Headquarters and "Headcount"

In [3]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import traceback
import time
import os
import pandas as pd

# Load credentials from environment variables
EMAIL = "bmorris@lacydiversified.com"
PASSWORD = "Oiaglobal25!"

# Setup Chrome options
options = Options()
options.add_argument("--start-maximized")

# Path to your ChromeDriver
service = Service(r'C:\Users\IlaBarshilia\Downloads\chromedriver-win64\chromedriver-win64\chromedriver.exe')  # 🔁 Replace with your actual path

# Start the browser
driver = webdriver.Chrome(service=service, options=options)
wait = WebDriverWait(driver, 15)

try:
    # Step 1: Open Capital IQ login page
    driver.get("https://www.capitaliq.spglobal.com")
    print("Opened Capital IQ login page.")

    # Step 2: Enter email
    email_input = wait.until(EC.presence_of_element_located((By.ID, "input28")))
    email_input.clear()
    email_input.send_keys(EMAIL)
    print("Email entered.")

    # Step 3: Click "Next"
    next_button = wait.until(EC.element_to_be_clickable((By.XPATH, '//input[@type="submit" and @value="Next"]')))
    next_button.click()
    print("Email submitted.")
    time.sleep(2)  # Wait for the password field to appear

    # Step 4: Enter password
    password_input = wait.until(EC.presence_of_element_located((By.XPATH, '//input[@placeholder="Password"]')))
    password_input.send_keys(PASSWORD)

    # Step 5: Click "Sign In"
    sign_in_button = wait.until(EC.element_to_be_clickable((By.XPATH, '//input[@type="submit" and @value="Sign In"]')))
    sign_in_button.click()
    print("Password submitted.")

    # Step 6: Wait for login to complete
    time.sleep(10)

    # Step 7: Navigate to company profile
    company_url = "https://www.capitaliq.spglobal.com/web/client#company/profile?id=15154262"
    driver.get(company_url)
    time.sleep(10)
    print("Navigated to company profile.")

except Exception as e:
    print("An error occurred:", e)
    traceback.print_exc()

# Initialize a dictionary to hold the extracted data
company_data = {
    "NAICS Code": None,
    "Industry": None,
    "Description": None,
    "Website URL": None,
    "Headquarters": None,
    "Current_Headcount": None,
    "Percent_Change_in_HC": None,
    "Change_in_HC": None
}

try:
    # 1. NAICS Code
    naics_element = wait.until(EC.presence_of_element_located(
        (By.XPATH, '//div[@id="naics"]//span[@id="naicsLessTextSpan"]/b')))
    company_data["NAICS Code"] = naics_element.text.strip()

    # 2. Industry
    industry_element = wait.until(EC.presence_of_element_located(
        (By.CSS_SELECTOR, 'h6[data-testid="company-industry-mini-body"]')))
    company_data["Industry"] = industry_element.text.strip()

    # 3. Description (after clicking 'VIEW ALL')
    view_all_button = wait.until(EC.element_to_be_clickable(
        (By.XPATH, '//a[contains(text(), "VIEW ALL")]')))
    view_all_button.click()
    time.sleep(1)

    description_element = wait.until(EC.presence_of_element_located(
        (By.ID, "compDesc")))
    description = description_element.text.strip()
    if "VIEW LESS" in description:
        description = description.split("VIEW LESS")[0].strip()
    company_data["Description"] = description

    # 4. Website URL
    cpweblink_element = wait.until(EC.presence_of_element_located(
        (By.CLASS_NAME, "cpweblink")))
    company_data["Website URL"] = cpweblink_element.find_element(By.TAG_NAME, "a").get_attribute("href")

    # 5. Headquarters
    block_label_element = wait.until(EC.presence_of_element_located(
        (By.CLASS_NAME, "blockLabel")))
    hq_element = block_label_element.find_element(By.XPATH, "following-sibling::*[1]")
    lines = hq_element.text.splitlines()
    company_data["Headquarters"] = ', '.join([line.strip() for line in lines if line.strip()])

    # 6. Headcount
    headcount_element = wait.until(EC.presence_of_element_located(
        (By.XPATH, '//div[@class="spg-d-flex"]/h6[@class="spg-heading spg-heading--xxsmall spg-text-nowrap"]')))
    company_data["Current_Headcount"] = headcount_element.text.strip()

    # 7. Last Year's Headcount
    percent_change = wait.until(EC.presence_of_element_located(
        (By.XPATH, '//strong[@class="spg-ml-xs"]')))
    percent_change = percent_change.text.strip().replace('%', '')
    company_data["Percent_Change_in_HC"] = percent_change

    change_type = wait.until(EC.presence_of_element_located((By.XPATH,
        '//div[@class="spg-d-flex spg-ml-sm spg-align-center spg-text-xsmall spg-px-xs css-vv7z8h"]//span[@data-icon="caret-up"]')))
    # Find the span with data-icon inside it
    change_type = change_type.get_attribute('outerHTML')
    if 'caret-up' in change_type:
        change_type = 1
    elif 'caret-down' in change_type:
        change_type = -1
    else:
        change_type = 0
    company_data["Change_in_HC"] = change_type
    # company_data["Last Year's Headcount"] = f"{percentage} {change_type.strip()}"
except Exception as e:
    print("An error occurred while extracting company data:", e)

# Close the browser
driver.quit()

# Create a DataFrame
df = pd.DataFrame([company_data])
df = df.astype({
    'Current_Headcount': 'int',
    'Change_in_HC': 'int',
    'Percent_Change_in_HC': 'float'
})
df['LY_Headcount'] = (df['Current_Headcount'] / (1 + df['Percent_Change_in_HC']/100)).round().astype('Int64')
df

Opened Capital IQ login page.
Email entered.
Email submitted.
Password submitted.
Navigated to company profile.


,NAICS Code,Industry,Description,Website URL,Headquarters,Current_Headcount,Percent_Change_in_HC,Change_in_HC,LY_Headcount
0,332999 - All Other Miscellaneous Fabricated Me...,Diversified Metals and Mining,Acme Barricades is a full-service highway safe...,http://www.acmebarricades.com/,"9800 Normandy Boulevard, Jacksonville, FL 3222...",62,5.0,1,59


#### Working - Full Code - navigates to search result and picks first company, loads all data

In [ ]:
import os
import time
import logging
import traceback
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Load credentials from environment variables (recommended)
EMAIL = "bmorris@lacydiversified.com"
PASSWORD = "Oiaglobal25!"

# Setup Chrome options
options = Options()
# options.add_argument("--headless")  # Uncomment if running in headless mode
driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 20)

try:
    # Step 1: Open Capital IQ login page
    driver.get("https://www.capitaliq.spglobal.com")
    logging.info("Opened Capital IQ login page.")

    # Step 2: Enter email
    email_input = wait.until(EC.presence_of_element_located((By.ID, "input28")))
    email_input.clear()
    email_input.send_keys(EMAIL)
    logging.info("Email entered.")

    # Step 3: Click "Next"
    next_button = wait.until(EC.element_to_be_clickable((By.XPATH, '//input[@type="submit" and @value="Next"]')))
    next_button.click()
    logging.info("Email submitted.")
    time.sleep(2)

    # Step 4: Enter password
    password_input = wait.until(EC.presence_of_element_located((By.XPATH, '//input[@placeholder="Password"]')))
    password_input.send_keys(PASSWORD)

    # Step 5: Click "Sign In"
    sign_in_button = wait.until(EC.element_to_be_clickable((By.XPATH, '//input[@type="submit" and @value="Sign In"]')))
    sign_in_button.click()
    logging.info("Password submitted.")

    # Step 6: Wait for login to complete
    time.sleep(10)

    # Step 7: Accept cookie popup if present
    try:
        cookie_button = WebDriverWait(driver, 5).until(
            EC.element_to_be_clickable((By.XPATH, '//button[contains(text(), "Accept") or contains(text(), "Agree")]'))
        )
        cookie_button.click()
        logging.info("Cookie popup accepted.")
    except:
        logging.info("No cookie popup found or already handled.")

    # Step 8: Wait for page to fully load
    wait.until(EC.presence_of_element_located((By.XPATH, '//div[contains(@class, "header")]')))
    logging.info("Page fully loaded.")

    # Step 9: Navigate directly to the search results URL
    search_term = "Acme Barricades"
    search_url = f"https://www.capitaliq.spglobal.com/apisv3/spg-webplatform-core/search/searchResults?vertical=&q={search_term.replace(' ', '%20')}"
    driver.get(search_url)
    logging.info(f"Navigated directly to search results for: {search_term}")
    time.sleep(5)  # Wait for search results to load
    # Step 10: Click on the first company link
    first_company_link = wait.until(EC.element_to_be_clickable((
        By.XPATH, '(//a[@data-testid="spg-link" and contains(@class, "css-1b9i013")])[1]')))
    driver.execute_script("arguments[0].click();", first_company_link)
    logging.info("Clicked on the first company link.")

except Exception as e:
    logging.error("An error occurred:")
    traceback.print_exc()
    driver.save_screenshot("error_debug.png")

# Initialize a dictionary to hold the extracted data
company_data = {
    "NAICS Code": None,
    "Industry": None,
    "Description": None,
    "Website URL": None,
    "Headquarters": None,
    "Current_Headcount": None,
    "Percent_Change_in_HC": None,
    "Change_in_HC": None
}

try:
    # 1. NAICS Code
    naics_element = wait.until(EC.presence_of_element_located(
        (By.XPATH, '//div[@id="naics"]//span[@id="naicsLessTextSpan"]/b')))
    company_data["NAICS Code"] = naics_element.text.strip()

    # 2. Industry
    industry_element = wait.until(EC.presence_of_element_located(
        (By.CSS_SELECTOR, 'h6[data-testid="company-industry-mini-body"]')))
    company_data["Industry"] = industry_element.text.strip()

    # 3. Description (after clicking 'VIEW ALL')
    view_all_button = wait.until(EC.element_to_be_clickable(
        (By.XPATH, '//a[contains(text(), "VIEW ALL")]')))
    view_all_button.click()
    time.sleep(1)

    description_element = wait.until(EC.presence_of_element_located(
        (By.ID, "compDesc")))
    description = description_element.text.strip()
    if "VIEW LESS" in description:
        description = description.split("VIEW LESS")[0].strip()
    company_data["Description"] = description

    # 4. Website URL
    cpweblink_element = wait.until(EC.presence_of_element_located(
        (By.CLASS_NAME, "cpweblink")))
    company_data["Website URL"] = cpweblink_element.find_element(By.TAG_NAME, "a").get_attribute("href")

    # 5. Headquarters
    block_label_element = wait.until(EC.presence_of_element_located(
        (By.CLASS_NAME, "blockLabel")))
    hq_element = block_label_element.find_element(By.XPATH, "following-sibling::*[1]")
    lines = hq_element.text.splitlines()
    company_data["Headquarters"] = ', '.join([line.strip() for line in lines if line.strip()])

    # 6. Headcount
    headcount_element = wait.until(EC.presence_of_element_located(
        (By.XPATH, '//div[@class="spg-d-flex"]/h6[@class="spg-heading spg-heading--xxsmall spg-text-nowrap"]')))
    company_data["Current_Headcount"] = headcount_element.text.strip()

    # 7. Last Year's Headcount
    percent_change = wait.until(EC.presence_of_element_located(
        (By.XPATH, '//strong[@class="spg-ml-xs"]')))
    percent_change = percent_change.text.strip().replace('%', '')
    company_data["Percent_Change_in_HC"] = percent_change

    change_type = wait.until(EC.presence_of_element_located((By.XPATH,
        '//div[@class="spg-d-flex spg-ml-sm spg-align-center spg-text-xsmall spg-px-xs css-vv7z8h"]//span[@data-icon="caret-up"]')))
    # Find the span with data-icon inside it
    change_type = change_type.get_attribute('outerHTML')
    if 'caret-up' in change_type:
        change_type = 1
    elif 'caret-down' in change_type:
        change_type = -1
    else:
        change_type = 0
    company_data["Change_in_HC"] = change_type
    # company_data["Last Year's Headcount"] = f"{percentage} {change_type.strip()}"
except Exception as e:
    print("An error occurred while extracting company data:", e)

# Close the browser
driver.quit()


# Create a DataFrame
df = pd.DataFrame([company_data])
df = df.astype({
    'Current_Headcount': 'int',
    'Change_in_HC': 'int',
    'Percent_Change_in_HC': 'float'
})

if (df['Current_Headcount'].isnull().any() or df['Percent_Change_in_HC'].isnull().any() or df['Change_in_HC'].isnull().any()):
    df['LY_Headcount'] = None
else:
    df['LY_Headcount'] = (df['Current_Headcount'] / (df['Change_in_HC']*df['Percent_Change_in_HC'])).round().astype('Int64')

df


2025-07-09 11:35:37,436 - INFO - Opened Capital IQ login page.
2025-07-09 11:35:46,752 - INFO - Email entered.
2025-07-09 11:35:46,846 - INFO - Email submitted.
2025-07-09 11:35:49,296 - INFO - Password submitted.
2025-07-09 11:36:00,886 - INFO - Cookie popup accepted.
2025-07-09 11:36:00,901 - INFO - Page fully loaded.
2025-07-09 11:36:01,939 - INFO - Navigated directly to search results for: Acme Barricades
2025-07-09 11:36:06,968 - INFO - Clicked on the first company link.


,NAICS Code,Industry,Description,Website URL,Headquarters,Current_Headcount,Percent_Change_in_HC,Change_in_HC,LY_Headcount
0,332999 - All Other Miscellaneous Fabricated Me...,,Acme Barricades is a full-service highway safe...,http://www.acmebarricades.com/,"9800 Normandy Boulevard, Jacksonville, FL 3222...",62,5.0,1,59


#### working for polygon - select first in global results

In [23]:
import os
import time
import logging
import traceback
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Setup logging
logging.basicConfig(
    level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S')

# Load credentials from environment variables (recommended)
EMAIL = "bmorris@lacydiversified.com"
PASSWORD = "Oiaglobal25!"

# Setup Chrome options
options = Options()
# options.add_argument("--headless")  # Uncomment if running in headless mode
driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 20)

try:
    # Step 1: Open Capital IQ login page
    driver.get("https://www.capitaliq.spglobal.com")
    logging.info("Opened Capital IQ login page.")

    # Step 2: Enter email
    email_input = wait.until(EC.presence_of_element_located((By.ID, "input28")))
    email_input.clear()
    email_input.send_keys(EMAIL)
    logging.info("Email entered.")

    # Step 3: Click "Next"
    next_button = wait.until(EC.element_to_be_clickable((By.XPATH, '//input[@type="submit" and @value="Next"]')))
    next_button.click()
    logging.info("Email submitted.")
    time.sleep(2)

    # Step 4: Enter password
    password_input = wait.until(EC.presence_of_element_located((By.XPATH, '//input[@placeholder="Password"]')))
    password_input.send_keys(PASSWORD)

    # Step 5: Click "Sign In"
    sign_in_button = wait.until(EC.element_to_be_clickable((By.XPATH, '//input[@type="submit" and @value="Sign In"]')))
    sign_in_button.click()
    logging.info("Password submitted.")

    # Step 6: Wait for login to complete
    time.sleep(10)

    # Step 7: Accept cookie popup if present
    try:
        cookie_button = WebDriverWait(driver, 5).until(
            EC.element_to_be_clickable((By.XPATH, '//button[contains(text(), "Accept") or contains(text(), "Agree")]')))
        cookie_button.click()
        logging.info("Cookie popup accepted.")
    except:
        logging.info("No cookie popup found or already handled.")

    # Step 8: Wait for page to fully load
    wait.until(EC.presence_of_element_located((By.XPATH, '//div[contains(@class, "header")]')))
    logging.info("Page fully loaded.")

    # Step 9: Navigate directly to the search results URL
    search_term = "Polygon Company"
    search_url = f"https://www.capitaliq.spglobal.com/apisv3/spg-webplatform-core/search/searchResults?vertical=&q={search_term.replace(' ', '%20')}"
    driver.get(search_url)
    logging.info(f"Navigated directly to search results for: {search_term}")
    time.sleep(5)  # Wait for search results to load

    # Step 10: Click on the first company link
    first_company_link = wait.until(EC.element_to_be_clickable((
        By.XPATH, '(//a[@data-testid="spg-link" and contains(@class, "css-1b9i013")])[1]')))
    driver.execute_script("arguments[0].click();", first_company_link)
    logging.info("Clicked on the first company link.")

    # Step 11: Accept cookie popup if it appears after navigating to the company page
    try:
        cookie_button = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.XPATH, '//button[contains(text(), "Accept") or contains(text(), "Agree")]'))
        )
        cookie_button.click()
        logging.info("Cookie popup accepted after navigating to company page.")
    except Exception as e:
        logging.info("No cookie popup found or already handled after company page navigation.")

except Exception as e:
    logging.error("An error occurred:")
    traceback.print_exc()
    driver.save_screenshot("error_debug.png")

# Initialize a dictionary to hold the extracted data
company_data = {
    "NAICS Code": None,
    "Industry": None,
    "Description": None,
    "Website URL": None,
    "Headquarters": None,
    "Current_Headcount": None,
    "Percent_Change_in_HC": None,
    "Change_in_HC": None,
    "CEO Name": None,
    "CEO Profile Link": None
}

# 1. NAICS Code
try:
    naics_element = wait.until(EC.presence_of_element_located(
        (By.XPATH, '//div[@id="naics"]//span[@id="naicsLessTextSpan"]/b')))
    company_data["NAICS Code"] = naics_element.text.strip()
except Exception as e:
    print("NAICS Code not found:", e)

# 2. Industry
try:
    industry_element = wait.until(EC.presence_of_element_located(
        (By.CSS_SELECTOR, 'h6[data-testid="company-industry-mini-body"]')))
    company_data["Industry"] = industry_element.text.strip()
except Exception as e:
    print("Industry not found!")

# 3. Description
try:
    view_all_button = wait.until(EC.element_to_be_clickable(
        (By.XPATH, '//a[contains(text(), "VIEW ALL")]')))
    view_all_button.click()
    time.sleep(1)
    description_element = wait.until(EC.presence_of_element_located((By.ID, "compDesc")))
    description = description_element.text.strip()
    if "VIEW LESS" in description:
        description = description.split("VIEW LESS")[0].strip()
    company_data["Description"] = description
except Exception as e:
    print("Description not found!")

# 4. Website URL
try:
    cpweblink_element = wait.until(EC.presence_of_element_located((By.CLASS_NAME, "cpweblink")))
    company_data["Website URL"] = cpweblink_element.find_element(By.TAG_NAME, "a").get_attribute("href")
except Exception as e:
    print("Website URL not found!")

# 5. Headquarters
try:
    block_label_element = wait.until(EC.presence_of_element_located((By.CLASS_NAME, "blockLabel")))
    hq_element = block_label_element.find_element(By.XPATH, "following-sibling::*[1]")
    lines = hq_element.text.splitlines()
    company_data["Headquarters"] = ', '.join([line.strip() for line in lines if line.strip()])
except Exception as e:
    print("Headquarters not found!")

# 6. Current Headcount
try:
    headcount_element = wait.until(EC.presence_of_element_located((
        By.XPATH, '//div[@class="spg-d-flex"]/h6[@class="spg-heading spg-heading--xxsmall spg-text-nowrap"]')))
    company_data["Current_Headcount"] = headcount_element.text.strip()
except Exception as e:
    print("Current Headcount not found!")

# 7. Percent Change in Headcount
try:
    percent_change = wait.until(EC.presence_of_element_located((By.XPATH, '//strong[@class="spg-ml-xs"]')))
    company_data["Percent_Change_in_HC"] = percent_change.text.strip().replace('%', '')
except Exception as e:
    print("Percent Change in Headcount not found!")

# 8. Change in Headcount Direction
try:
    change_type = wait.until(EC.presence_of_element_located((By.XPATH,
        '//div[contains(@class, "spg-d-flex") and contains(@class, "spg-align-center")]//span[@data-icon]')))
    icon_html = change_type.get_attribute('outerHTML')
    if 'caret-up' in icon_html:
        company_data["Change_in_HC"] = 1
    elif 'caret-down' in icon_html:
        company_data["Change_in_HC"] = -1
    else:
        company_data["Change_in_HC"] = 0
except Exception as e:
    print("Change in Headcount direction not found!")

# 9. CEO Name and Profile Link
try:
    ceo_element = wait.until(EC.presence_of_element_located((
        By.CSS_SELECTOR, 'a[data-testid="ribbon-ceo"].spg-text-medium.css-12tcig2.css-1b9i013')))
    company_data["CEO Name"] = ceo_element.text
    company_data["CEO Profile Link"] = ceo_element.get_attribute("href")
except Exception as e:
    print("CEO information not found!")

# Close the browser
driver.quit()

# Create a DataFrame
df2 = pd.DataFrame([company_data])

# Convert columns to numeric, errors='coerce' will turn invalid parsing into NaN
for col, dtype in {
    'Current_Headcount': 'Int64',
    'Change_in_HC': 'Int64',
    'Percent_Change_in_HC': 'float'}.items():
    df2[col] = pd.to_numeric(df2[col], errors='coerce').astype(dtype)

if (
    df2['Current_Headcount'].isnull().any() or
    df2['Percent_Change_in_HC'].isnull().any() or
    df2['Change_in_HC'].isnull().any()):
    df2['LY_Headcount'] = pd.NA
else:
    df2['LY_Headcount'] = (df2['Current_Headcount'] / (1 + (df2['Change_in_HC']*df2['Percent_Change_in_HC'])/100)).round().astype('Int64')
# Reorder columns to bring LY_Headcount after Change_in_HC
cols = list(df2.columns)
if "LY_Headcount" in cols and "Change_in_HC" in cols:
    ly_idx = cols.index("LY_Headcount")
    chg_idx = cols.index("Change_in_HC")
    # Remove LY_Headcount from its current position
    cols.pop(ly_idx)
    # Insert LY_Headcount after Change_in_HC
    cols.insert(chg_idx + 1, "LY_Headcount")
    df2 = df2[cols]
df2

2025-07-09 14:31:08,791 - INFO - Opened Capital IQ login page.
2025-07-09 14:31:16,823 - INFO - Email entered.
2025-07-09 14:31:17,008 - INFO - Email submitted.
2025-07-09 14:31:19,685 - INFO - Password submitted.
2025-07-09 14:31:35,382 - INFO - No cookie popup found or already handled.
2025-07-09 14:31:35,426 - INFO - Page fully loaded.
2025-07-09 14:31:36,826 - INFO - Navigated directly to search results for: Polygon Company
2025-07-09 14:31:42,880 - INFO - Clicked on the first company link.
2025-07-09 14:31:46,861 - INFO - Cookie popup accepted after navigating to company page.


Website URL not found!
Current Headcount not found!
Percent Change in Headcount not found!


,NAICS Code,Industry,Description,Website URL,Headquarters,Current_Headcount,Percent_Change_in_HC,Change_in_HC,LY_Headcount,CEO Name,CEO Profile Link
0,237210 - Land Subdivision,Real Estate Development,Polygon Real Estate Ltd operates in real estat...,None,"Granite 8, Kiryat Arieh, PO Box 10033, Petah T...",<NA>,NaN,1,<NA>,Nir Yohai Shochat,https://www.capitaliq.spglobal.com/web/client#...


#### Working - select first result in private companies

In [25]:
import os
import time
import logging
import traceback
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Setup logging
logging.basicConfig(
    level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S')

# Load credentials from environment variables (recommended)
EMAIL = "bmorris@lacydiversified.com"
PASSWORD = "Oiaglobal25!"

# Setup Chrome options
options = Options()
# options.add_argument("--headless")  # Uncomment if running in headless mode
driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 20)

try:
    # Step 1: Open Capital IQ login page
    driver.get("https://www.capitaliq.spglobal.com")
    logging.info("Opened Capital IQ login page.")

    # Step 2: Enter email
    email_input = wait.until(EC.presence_of_element_located((By.ID, "input28")))
    email_input.clear()
    email_input.send_keys(EMAIL)
    logging.info("Email entered.")

    # Step 3: Click "Next"
    next_button = wait.until(EC.element_to_be_clickable((By.XPATH, '//input[@type="submit" and @value="Next"]')))
    next_button.click()
    logging.info("Email submitted.")
    time.sleep(2)

    # Step 4: Enter password
    password_input = wait.until(EC.presence_of_element_located((By.XPATH, '//input[@placeholder="Password"]')))
    password_input.send_keys(PASSWORD)

    # Step 5: Click "Sign In"
    sign_in_button = wait.until(EC.element_to_be_clickable((By.XPATH, '//input[@type="submit" and @value="Sign In"]')))
    sign_in_button.click()
    logging.info("Password submitted.")

    # Step 6: Wait for login to complete
    time.sleep(10)

    # Step 7: Accept cookie popup if present
    try:
        cookie_button = WebDriverWait(driver, 5).until(
            EC.element_to_be_clickable((By.XPATH, '//button[contains(text(), "Accept") or contains(text(), "Agree")]')))
        cookie_button.click()
        logging.info("Cookie popup accepted.")
    except:
        logging.info("No cookie popup found or already handled.")

    # Step 8: Wait for page to fully load
    wait.until(EC.presence_of_element_located((By.XPATH, '//div[contains(@class, "header")]')))
    logging.info("Page fully loaded.")

    # Step 9: Navigate directly to the search results URL
    search_term = "Polygon Company"
    search_url = f"https://www.capitaliq.spglobal.com/apisv3/spg-webplatform-core/search/searchResults?vertical=&q={search_term.replace(' ', '%20')}"
    driver.get(search_url)
    logging.info(f"Navigated directly to search results for: {search_term}")
    time.sleep(5)  # Wait for search results to load

    # Step 10: Click on the first company link
    private_company_link = wait.until(EC.element_to_be_clickable((
        By.XPATH, '//h4[text()="Private Companies"]/following::a[@data-testid="spg-link" and contains(@class, "css-1b9i013")][1]')))
    driver.execute_script("arguments[0].click();", private_company_link)
    logging.info("Clicked on the first private company link.")

    # Step 11: Accept cookie popup if it appears after navigating to the company page
    try:
        cookie_button = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.XPATH, '//button[contains(text(), "Accept") or contains(text(), "Agree")]'))
        )
        cookie_button.click()
        logging.info("Cookie popup accepted after navigating to company page.")
    except Exception as e:
        logging.info("No cookie popup found or already handled after company page navigation.")

except Exception as e:
    logging.error("An error occurred:")
    traceback.print_exc()
    driver.save_screenshot("error_debug.png")

# Initialize a dictionary to hold the extracted data
company_data = {
    "NAICS Code": None,
    "Industry": None,
    "Description": None,
    "Website URL": None,
    "Headquarters": None,
    "Current_Headcount": None,
    "Percent_Change_in_HC": None,
    "Change_in_HC": None,
    "CEO Name": None,
    "CEO Profile Link": None
}

# 1. NAICS Code
try:
    naics_element = wait.until(EC.presence_of_element_located(
        (By.XPATH, '//div[@id="naics"]//span[@id="naicsLessTextSpan"]/b')))
    company_data["NAICS Code"] = naics_element.text.strip()
except Exception as e:
    print("NAICS Code not found:", e)

# 2. Industry
try:
    industry_element = wait.until(EC.presence_of_element_located(
        (By.CSS_SELECTOR, 'h6[data-testid="company-industry-mini-body"]')))
    company_data["Industry"] = industry_element.text.strip()
except Exception as e:
    print("Industry not found!")

# 3. Description
try:
    view_all_button = wait.until(EC.element_to_be_clickable(
        (By.XPATH, '//a[contains(text(), "VIEW ALL")]')))
    view_all_button.click()
    time.sleep(1)
    description_element = wait.until(EC.presence_of_element_located((By.ID, "compDesc")))
    description = description_element.text.strip()
    if "VIEW LESS" in description:
        description = description.split("VIEW LESS")[0].strip()
    company_data["Description"] = description
except Exception as e:
    print("Description not found!")

# 4. Website URL
try:
    cpweblink_element = wait.until(EC.presence_of_element_located((By.CLASS_NAME, "cpweblink")))
    company_data["Website URL"] = cpweblink_element.find_element(By.TAG_NAME, "a").get_attribute("href")
except Exception as e:
    print("Website URL not found!")

# 5. Headquarters
try:
    block_label_element = wait.until(EC.presence_of_element_located((By.CLASS_NAME, "blockLabel")))
    hq_element = block_label_element.find_element(By.XPATH, "following-sibling::*[1]")
    lines = hq_element.text.splitlines()
    company_data["Headquarters"] = ', '.join([line.strip() for line in lines if line.strip()])
except Exception as e:
    print("Headquarters not found!")

# 6. Current Headcount
try:
    headcount_element = wait.until(EC.presence_of_element_located((
        By.XPATH, '//div[@class="spg-d-flex"]/h6[@class="spg-heading spg-heading--xxsmall spg-text-nowrap"]')))
    company_data["Current_Headcount"] = headcount_element.text.strip()
except Exception as e:
    print("Current Headcount not found!")

# 7. Percent Change in Headcount
try:
    percent_change = wait.until(EC.presence_of_element_located((By.XPATH, '//strong[@class="spg-ml-xs"]')))
    company_data["Percent_Change_in_HC"] = percent_change.text.strip().replace('%', '')
except Exception as e:
    print("Percent Change in Headcount not found!")

# 8. Change in Headcount Direction
try:
    change_type = wait.until(EC.presence_of_element_located((By.XPATH,
        '//div[contains(@class, "spg-d-flex") and contains(@class, "spg-align-center")]//span[@data-icon]')))
    icon_html = change_type.get_attribute('outerHTML')
    if 'caret-up' in icon_html:
        company_data["Change_in_HC"] = 1
    elif 'caret-down' in icon_html:
        company_data["Change_in_HC"] = -1
    else:
        company_data["Change_in_HC"] = 0
except Exception as e:
    print("Change in Headcount direction not found!")

# 9. CEO Name and Profile Link
try:
    ceo_element = wait.until(EC.presence_of_element_located((
        By.CSS_SELECTOR, 'a[data-testid="ribbon-ceo"].spg-text-medium.css-12tcig2.css-1b9i013')))
    company_data["CEO Name"] = ceo_element.text
    company_data["CEO Profile Link"] = ceo_element.get_attribute("href")
except Exception as e:
    print("CEO information not found!")

# Close the browser
driver.quit()

# Create a DataFrame
df2 = pd.DataFrame([company_data])

# Convert columns to numeric, errors='coerce' will turn invalid parsing into NaN
for col, dtype in {
    'Current_Headcount': 'Int64',
    'Change_in_HC': 'Int64',
    'Percent_Change_in_HC': 'float'}.items():
    df2[col] = pd.to_numeric(df2[col], errors='coerce').astype(dtype)

if (
    df2['Current_Headcount'].isnull().any() or
    df2['Percent_Change_in_HC'].isnull().any() or
    df2['Change_in_HC'].isnull().any()):
    df2['LY_Headcount'] = pd.NA
else:
    df2['LY_Headcount'] = (df2['Current_Headcount'] / (1 + (df2['Change_in_HC']*df2['Percent_Change_in_HC'])/100)).round().astype('Int64')
# Reorder columns to bring LY_Headcount after Change_in_HC
cols = list(df2.columns)
if "LY_Headcount" in cols and "Change_in_HC" in cols:
    ly_idx = cols.index("LY_Headcount")
    chg_idx = cols.index("Change_in_HC")
    # Remove LY_Headcount from its current position
    cols.pop(ly_idx)
    # Insert LY_Headcount after Change_in_HC
    cols.insert(chg_idx + 1, "LY_Headcount")
    df2 = df2[cols]
df2

2025-07-09 14:42:51,333 - INFO - Opened Capital IQ login page.
2025-07-09 14:42:59,804 - INFO - Email entered.
2025-07-09 14:42:59,981 - INFO - Email submitted.
2025-07-09 14:43:02,577 - INFO - Password submitted.
2025-07-09 14:43:16,211 - INFO - Cookie popup accepted.
2025-07-09 14:43:16,240 - INFO - Page fully loaded.
2025-07-09 14:43:17,563 - INFO - Navigated directly to search results for: Polygon Company
2025-07-09 14:43:22,788 - INFO - Clicked on the first private company link.
2025-07-09 14:43:33,173 - INFO - No cookie popup found or already handled after company page navigation.


,NAICS Code,Industry,Description,Website URL,Headquarters,Current_Headcount,Percent_Change_in_HC,Change_in_HC,LY_Headcount,CEO Name,CEO Profile Link
0,326121 - Unlaminated Plastics Profile Shape Ma...,Commodity Chemicals,"Polygon Company, Inc., an engineered materials...",http://www.polygoncomposites.com/,"103 Industrial Park Drive, PO Box 176, Walkert...",71,8.0,-1,77,Bree Katulak,https://www.capitaliq.spglobal.com/web/client#...


#### working - to get LinkedIN Professionals

In [ ]:
import os
import time
import logging
import traceback
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Setup logging
logging.basicConfig(
    level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
    force=True)  # Force logging to overwrite any previous settings

# Load credentials from environment variables (recommended)
EMAIL = "bmorris@lacydiversified.com"
PASSWORD = "Oiaglobal25!"

# Setup Chrome options
options = Options()
# options.add_argument("--headless")  # Uncomment if running in headless mode
driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 20)

try:
    # Step 1: Open Capital IQ login page
    driver.get("https://www.capitaliq.spglobal.com")
    logging.info("Opened Capital IQ login page.")

    # Step 2: Enter email
    email_input = wait.until(EC.presence_of_element_located((By.ID, "input28")))
    email_input.clear()
    email_input.send_keys(EMAIL)
    logging.info("Email entered.")

    # Step 3: Click "Next"
    next_button = wait.until(EC.element_to_be_clickable((By.XPATH, '//input[@type="submit" and @value="Next"]')))
    next_button.click()
    logging.info("Email submitted.")
    time.sleep(2)

    # Step 4: Enter password
    password_input = wait.until(EC.presence_of_element_located((By.XPATH, '//input[@placeholder="Password"]')))
    password_input.send_keys(PASSWORD)

    # Step 5: Click "Sign In"
    sign_in_button = wait.until(EC.element_to_be_clickable((By.XPATH, '//input[@type="submit" and @value="Sign In"]')))
    sign_in_button.click()
    logging.info("Password submitted.")

    # Step 6: Wait for login to complete
    time.sleep(10)

    # Step 7: Accept cookie popup if present
    try:
        cookie_button = WebDriverWait(driver, 5).until(
            EC.element_to_be_clickable((By.XPATH, '//button[contains(text(), "Accept") or contains(text(), "Agree")]')))
        cookie_button.click()
        logging.info("Cookie popup accepted.")
    except:
        logging.info("No cookie popup found or already handled.")

    # Step 8: Wait for page to fully load
    wait.until(EC.presence_of_element_located((By.XPATH, '//div[contains(@class, "header")]')))
    logging.info("Page fully loaded.")

    # Step 9: Navigate directly to the search results URL
    search_term = "Polygon Company"
    search_city = "Walkerton"
    search_url = f"https://www.capitaliq.spglobal.com/apisv3/spg-webplatform-core/search/searchResults?vertical=&q={search_term.replace(' ', '%20')}"
    driver.get(search_url)
    logging.info(f"Navigated directly to search results for: {search_term}")
    time.sleep(5)  # Wait for search results to load

    # Step 10: Click on the first company link
    private_company_link = wait.until(EC.element_to_be_clickable((
        By.XPATH, '//h4[text()="Private Companies"]/following::a[@data-testid="spg-link" and contains(@class, "css-1b9i013")][1]')))
    driver.execute_script("arguments[0].click();", private_company_link)
    logging.info("Clicked on the first private company link.")

    # Step 11: Accept cookie popup if it appears after navigating to the company page
    try:
        cookie_button = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.XPATH, '//button[contains(text(), "Accept") or contains(text(), "Agree")]'))
        )
        cookie_button.click()
        logging.info("Cookie popup accepted after navigating to company page.")
    except Exception as e:
        logging.info("No cookie popup found or already handled after company page navigation.")

except Exception as e:
    logging.error("An error occurred:")
    traceback.print_exc()
    driver.save_screenshot("error_debug.png")

# Initialize a dictionary to hold the extracted data
company_data = {
    "NAICS Code": None,
    "Industry": None,
    "Description": None,
    "Website URL": None,
    "Headquarters": None,
    "Current_Headcount": None,
    "Percent_Change_in_HC": None,
    "Change_in_HC": None,
    "CEO Name": None,
    "CEO Profile Link": None,
    "Key People": None
}

# 1. NAICS Code
try:
    naics_element = wait.until(EC.presence_of_element_located(
        (By.XPATH, '//div[@id="naics"]//span[@id="naicsLessTextSpan"]/b')))
    company_data["NAICS Code"] = naics_element.text.strip()
except Exception as e:
    print("NAICS Code not found:", e)

# 2. Industry
try:
    industry_element = wait.until(EC.presence_of_element_located(
        (By.CSS_SELECTOR, 'h6[data-testid="company-industry-mini-body"]')))
    company_data["Industry"] = industry_element.text.strip()
except Exception as e:
    print("Industry not found!")

# 3. Description
try:
    view_all_button = wait.until(EC.element_to_be_clickable(
        (By.XPATH, '//a[contains(text(), "VIEW ALL")]')))
    view_all_button.click()
    time.sleep(1)
    description_element = wait.until(EC.presence_of_element_located((By.ID, "compDesc")))
    description = description_element.text.strip()
    if "VIEW LESS" in description:
        description = description.split("VIEW LESS")[0].strip()
    company_data["Description"] = description
except Exception as e:
    print("Description not found!")

# 4. Website URL
try:
    cpweblink_element = wait.until(EC.presence_of_element_located((By.CLASS_NAME, "cpweblink")))
    company_data["Website URL"] = cpweblink_element.find_element(By.TAG_NAME, "a").get_attribute("href")
except Exception as e:
    print("Website URL not found!")

# 5. Headquarters
try:
    block_label_element = wait.until(EC.presence_of_element_located((By.CLASS_NAME, "blockLabel")))
    hq_element = block_label_element.find_element(By.XPATH, "following-sibling::*[1]")
    lines = hq_element.text.splitlines()
    company_data["Headquarters"] = ', '.join([line.strip() for line in lines if line.strip()])
except Exception as e:
    print("Headquarters not found!")

# 6. Current Headcount
try:
    headcount_element = wait.until(EC.presence_of_element_located((
        By.XPATH, '//div[@class="spg-d-flex"]/h6[@class="spg-heading spg-heading--xxsmall spg-text-nowrap"]')))
    company_data["Current_Headcount"] = headcount_element.text.strip()
except Exception as e:
    print("Current Headcount not found!")

# 7. Percent Change in Headcount
try:
    percent_change = wait.until(EC.presence_of_element_located((By.XPATH, '//strong[@class="spg-ml-xs"]')))
    company_data["Percent_Change_in_HC"] = percent_change.text.strip().replace('%', '')
except Exception as e:
    print("Percent Change in Headcount not found!")

# 8. Change in Headcount Direction
try:
    change_type = wait.until(EC.presence_of_element_located((By.XPATH,
        '//div[contains(@class, "spg-d-flex") and contains(@class, "spg-align-center")]//span[@data-icon]')))
    icon_html = change_type.get_attribute('outerHTML')
    if 'caret-up' in icon_html:
        company_data["Change_in_HC"] = 1
    elif 'caret-down' in icon_html:
        company_data["Change_in_HC"] = -1
    else:
        company_data["Change_in_HC"] = 0
except Exception as e:
    print("Change in Headcount direction not found!")

# 9. CEO Name and Profile Link
try:
    ceo_element = wait.until(EC.presence_of_element_located((
        By.CSS_SELECTOR, 'a[data-testid="ribbon-ceo"].spg-text-medium.css-12tcig2.css-1b9i013')))
    company_data["CEO Name"] = ceo_element.text
    company_data["CEO Profile Link"] = ceo_element.get_attribute("href")
except Exception as e:
    print("CEO information not found!")

# 10. Officers
# Click the "More" link under Officers & Directors
more_button = wait.until(EC.element_to_be_clickable((By.XPATH, '//div[@id="offDirectors"]//a[contains(text(), "More")]')))
driver.execute_script("arguments[0].click();", more_button)
logging.info("Clicked on 'More' under Officers & Directors.")
time.sleep(5)

# Step 2: Wait for the table to appear
try:
    wait.until(EC.presence_of_element_located((By.ID, "section_1_control_23_section_0_control_71_i2_grid_table")))
    table = driver.find_element(By.ID, 'section_1_control_23_section_0_control_71_i2_grid_table')
except Exception as e:
    logging.error(f"Table not found: {e}")
    driver.quit()
    exit()

# Step 3: Loop through rows and print all <td> contents
rows = table.find_elements(By.XPATH, './/tbody/tr')
merged_data = []
for i, row in enumerate(rows):
    try:
        name = row.find_element(By.XPATH, './/td[@data-colkey="_NameFromDoc"]').text.strip()
        title = row.find_element(By.XPATH, './/td[@data-colkey="_Title"]/div').text.strip()
        merged_data.append(f" {title} : {name if name else '[No Name]'}")
        print(f"Row {i+1} — Name: {name if name else '[No Name]'}, Title: {title}")
    except Exception as e:
        logging.warning(f"Could not process row {i+1}: {e}")
final_output = '; '.join(merged_data)
company_data["Key People"] = final_output

# Optional: Close the browser
# driver.quit()


# Close the browser
# driver.quit()

# Create a DataFrame
df2 = pd.DataFrame([company_data])

# Convert columns to numeric, errors='coerce' will turn invalid parsing into NaN
for col, dtype in {
    'Current_Headcount': 'Int64',
    'Change_in_HC': 'Int64',
    'Percent_Change_in_HC': 'float'}.items():
    df2[col] = pd.to_numeric(df2[col], errors='coerce').astype(dtype)

if (
    df2['Current_Headcount'].isnull().any() or
    df2['Percent_Change_in_HC'].isnull().any() or
    df2['Change_in_HC'].isnull().any()):
    df2['LY_Headcount'] = pd.NA
else:
    df2['LY_Headcount'] = (df2['Current_Headcount'] / (1 + (df2['Change_in_HC']*df2['Percent_Change_in_HC'])/100)).round().astype('Int64')
# Reorder columns to bring LY_Headcount after Change_in_HC
cols = list(df2.columns)
if "LY_Headcount" in cols and "Change_in_HC" in cols:
    ly_idx = cols.index("LY_Headcount")
    chg_idx = cols.index("Change_in_HC")
    # Remove LY_Headcount from its current position
    cols.pop(ly_idx)
    # Insert LY_Headcount after Change_in_HC
    cols.insert(chg_idx + 1, "LY_Headcount")
    df2 = df2[cols]
df2

2025-07-09 17:47:00 - INFO - Opened Capital IQ login page.
2025-07-09 17:47:09 - INFO - Email entered.
2025-07-09 17:47:09 - INFO - Email submitted.
2025-07-09 17:47:12 - INFO - Password submitted.
2025-07-09 17:47:30 - INFO - Cookie popup accepted.
2025-07-09 17:47:30 - INFO - Page fully loaded.
2025-07-09 17:47:32 - INFO - Navigated directly to search results for: Polygon Company
2025-07-09 17:47:39 - INFO - Clicked on the first private company link.
2025-07-09 17:47:49 - INFO - No cookie popup found or already handled after company page navigation.
2025-07-09 17:47:52 - INFO - Clicked on 'More' under Officers & Directors.


Row 1 — Name: Bree Katulak, Title: President & CEO
Row 2 — Name: Jim Hoffman, Title: Vice President of Finance & IT
Row 3 — Name: Ben Fouch, Title: Vice President of Strategy & Operations
Row 4 — Name: Doug Drummond, Title: Vice President of Sales & Marketing


,NAICS Code,Industry,Description,Website URL,Headquarters,Current_Headcount,Percent_Change_in_HC,Change_in_HC,LY_Headcount,CEO Name,CEO Profile Link,Key People
0,326121 - Unlaminated Plastics Profile Shape Ma...,Commodity Chemicals,"Polygon Company, Inc., an engineered materials...",http://www.polygoncomposites.com/,"103 Industrial Park Drive, PO Box 176, Walkert...",71,8.0,-1,77,Bree Katulak,https://www.capitaliq.spglobal.com/web/client#...,President & CEO : Bree Katulak; Vice Preside...


#### Trial - get text from search results and only click once city matches

In [17]:
import os
import time
import logging
import traceback
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains


# Setup logging
logging.basicConfig(
    level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
    force=True)  # Force logging to overwrite any previous settings

# Load credentials from environment variables (recommended)
EMAIL = "bmorris@lacydiversified.com"
PASSWORD = "Oiaglobal25!"

# Setup Chrome options
options = Options()
# options.add_argument("--headless")  # Uncomment if running in headless mode
driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 20)

# Step 1: Open Capital IQ login page
driver.get("https://www.capitaliq.spglobal.com")
logging.info("Opened Capital IQ login page.")

# Step 2: Enter email
email_input = wait.until(EC.presence_of_element_located((By.ID, "input28")))
email_input.clear()
email_input.send_keys(EMAIL)
logging.info("Email entered.")

# Step 3: Click "Next"
next_button = wait.until(EC.element_to_be_clickable((By.XPATH, '//input[@type="submit" and @value="Next"]')))
next_button.click()
logging.info("Email submitted.")
time.sleep(2)

# Step 4: Enter password
password_input = wait.until(EC.presence_of_element_located((By.XPATH, '//input[@placeholder="Password"]')))
password_input.send_keys(PASSWORD)

# Step 5: Click "Sign In"
sign_in_button = wait.until(EC.element_to_be_clickable((By.XPATH, '//input[@type="submit" and @value="Sign In"]')))
sign_in_button.click()
logging.info("Password submitted.")

# Step 6: Wait for login to complete
time.sleep(10)

# Step 7: Accept cookie popup if present
try:
    cookie_button = WebDriverWait(driver, 5).until(
        EC.element_to_be_clickable((By.XPATH, '//button[contains(text(), "Accept") or contains(text(), "Agree")]')))
    cookie_button.click()
    logging.info("Cookie popup accepted.")
except:
    logging.info("No cookie popup found or already handled.")

# Step 8: Wait for page to fully load
wait.until(EC.presence_of_element_located((By.XPATH, '//div[contains(@class, "header")]')))
logging.info("Page fully loaded.")

# Step 9: Navigate directly to the search results URL
search_term = "Polygon Company"
search_city = "Walkerton"
search_url = f"https://www.capitaliq.spglobal.com/apisv3/spg-webplatform-core/search/searchResults?q={search_term.replace(' ', '%20')}&vertical=private_companies_mi-gss"
driver.get(search_url)
logging.info(f"Navigated directly to Private Companies search results for: {search_term}")
time.sleep(5)

# Step 10a: Change results per page to 50 using ActionChains
try:
    # Scroll to the dropdown area
    show_label = wait.until(EC.presence_of_element_located((By.CLASS_NAME, "css-7mog2u")))
    driver.execute_script("arguments[0].scrollIntoView(true);", show_label)
    time.sleep(1)

    # Click the dropdown trigger
    dropdown_trigger = wait.until(EC.element_to_be_clickable((By.XPATH, '//button[@type="button" and @title="20"]')))
    driver.execute_script("arguments[0].click();", dropdown_trigger)
    time.sleep(1)    
    options = wait.until(EC.presence_of_all_elements_located((By.XPATH, '//div[@title="50" and contains(@class, "css-11d71qm")]')))
    clicked = False
    for option in options:
            if option.is_displayed():
                driver.execute_script("arguments[0].click();", option)
                time.sleep(3)
                clicked = True
                break

    if not clicked:
        print("⚠️ Option with title='50' was not visible.")
except Exception as e:
    logging.error(f"Error while changing results per page: {e}")


# Step 10b: Wait for company cards to load
search_term = "Walkerton"

try:
    wait = WebDriverWait(driver, 20)

    # Try locating the table by class
    try:
        table = wait.until(EC.visibility_of_element_located(
            (By.XPATH, '//table[contains(@class, "css-rt2ub6") and contains(@class, "css-146ti0r")]')
        ))
        rows = table.find_elements(By.XPATH, './/tbody/tr')
        print("✅ Table found using class.")
    except:
        # Fallback: locate rows using divs with data-testid and class
        rows = wait.until(EC.presence_of_all_elements_located(
            (By.XPATH, '//div[@data-testid="grid-card" and contains(@class, "css-jqzb9n")]')
        ))
        print("✅ Rows found using grid-card divs.")

    for index, row in enumerate(rows):
        try:
            if 'table' in locals():
                cols = row.find_elements(By.XPATH, './/td')
                if len(cols) < 4:
                    continue  # Skip rows with fewer than 4 columns
                geography_text = cols[3].text.strip()
            else:
                geography_text = row.text.strip()

            if search_term.lower() in geography_text.lower():
                if 'table' in locals():
                    company_link = cols[0].find_element(By.XPATH, './/a')
                else:
                    company_link = row.find_element(By.XPATH, './/a')

                print(f"🔗 Clicking company: {company_link.text.strip()}")
                company_link.click()
                break
        except Exception as e:
            print(f"⚠️ Error processing row {index + 1}: {e}")

except Exception as e:
    print(f"❌ Error locating or processing company rows: {e}")

2025-07-14 14:33:32 - INFO - Opened Capital IQ login page.
2025-07-14 14:33:38 - INFO - Email entered.
2025-07-14 14:33:38 - INFO - Email submitted.
2025-07-14 14:33:40 - INFO - Password submitted.
2025-07-14 14:33:52 - INFO - Cookie popup accepted.
2025-07-14 14:33:52 - INFO - Page fully loaded.
2025-07-14 14:33:53 - INFO - Navigated directly to Private Companies search results for: Polygon Company


✅ Rows found using grid-card divs.


#### Working - Searches for company, then city then navigates among first 50 records

In [3]:
import os
import time
import logging
import traceback
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains


# Setup logging
logging.basicConfig(
    level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
    force=True)  # Force logging to overwrite any previous settings

# Load credentials from environment variables (recommended)
EMAIL = "bmorris@lacydiversified.com"
PASSWORD = "Oiaglobal25!"

# Setup Chrome options
options = Options()
# options.add_argument("--headless")  # Uncomment if running in headless mode
driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 20)

# Step 1: Open Capital IQ login page
driver.get("https://www.capitaliq.spglobal.com")
logging.info("Opened Capital IQ login page.")

# Step 2: Enter email
email_input = wait.until(EC.presence_of_element_located((By.ID, "input28")))
email_input.clear()
email_input.send_keys(EMAIL)
logging.info("Email entered.")

# Step 3: Click "Next"
next_button = wait.until(EC.element_to_be_clickable((By.XPATH, '//input[@type="submit" and @value="Next"]')))
next_button.click()
logging.info("Email submitted.")

# Step 4: Enter password
password_input = wait.until(EC.presence_of_element_located((By.XPATH, '//input[@placeholder="Password"]')))
password_input.send_keys(PASSWORD)

# Step 5: Click "Sign In"
sign_in_button = wait.until(EC.element_to_be_clickable((By.XPATH, '//input[@type="submit" and @value="Sign In"]')))
sign_in_button.click()
logging.info("Password submitted.")

# Step 6: Wait for login to complete
wait.until(EC.presence_of_element_located((By.XPATH, '//div[contains(@class, "header")]')))
logging.info("Login successful and page loaded.")

# Step 7: Accept cookie popup if present
try:
    cookie_button = WebDriverWait(driver, 10).until(
        EC.element_to_be_clickable((By.XPATH, '//button[contains(text(), "Accept") or contains(text(), "Agree")]')))
    cookie_button.click()
    logging.info("Cookie popup accepted.")
except:
    logging.info("No cookie popup found or already handled.")

# Step 8: Navigate directly to the search results URL
search_term = "Polygon Co"
search_city = "Walkerton"
search_url = f"https://www.capitaliq.spglobal.com/apisv3/spg-webplatform-core/search/searchResults?q={search_term.replace(' ', '%20')}&vertical=private_companies_mi-gss"
driver.get(search_url)
logging.info(f"Navigated to search results for: {search_term}")
wait.until(EC.presence_of_element_located((By.CLASS_NAME, "css-7mog2u")))


# Step 8.1: Handle cookie popup if present
try:
    cookie_button = WebDriverWait(driver, 10).until(
        EC.element_to_be_clickable((By.XPATH, '//button[contains(text(), "Accept") or contains(text(), "Agree")]')))
    cookie_button.click()
    logging.info("Cookie popup accepted.")
except:
    logging.info("No cookie popup found or already handled.")

# Step 9: Change results per page to 50
try:
    show_label = wait.until(EC.presence_of_element_located((By.CLASS_NAME, "css-7mog2u")))
    driver.execute_script("arguments[0].scrollIntoView(true);", show_label)
    time.sleep(1)

    dropdown_trigger = wait.until(EC.element_to_be_clickable((By.XPATH, '//button[@type="button" and @title="20"]')))
    driver.execute_script("arguments[0].click();", dropdown_trigger)
    time.sleep(1)

    options = wait.until(EC.presence_of_all_elements_located((By.XPATH, '//div[@title="50" and contains(@class, "css-11d71qm")]')))
    for option in options:
        if option.is_displayed():
            driver.execute_script("arguments[0].click();", option)
            time.sleep(3)
            break
except Exception as e:
    logging.error(f"Error while changing results per page: {e}")

# Step 10: Wait for company cards to load and click matching company
try:
    WebDriverWait(driver, 10).until(
        EC.presence_of_all_elements_located((By.XPATH, '//a[contains(@href, "/web/client#company/profile?id=")]'))
    )

    company_links = driver.find_elements(By.XPATH, '//a[contains(@href, "/web/client#company/profile?id=")]')
    match_found = False

    for link in company_links:
        try:
            # Go up to the grandparent container
            container = link.find_element(By.XPATH, './ancestor::div[3]')
            container_text = container.text

            if search_city.lower() in container_text.lower():
                print(f"✅ Match found: {link.text.strip()} in {search_city}")
                driver.execute_script("arguments[0].click();", link)
                match_found = True
                break

        except Exception as e:
            print(f"⚠️ Error processing link: {e}")

    if not match_found:
        print(f"❗ No matching company found for city: {search_city}")

except Exception as e:
    print(f"❌ Error locating company links: {e}")

# Initialize a dictionary to hold the extracted data
company_data = {
    "NAICS Code": None,
    "Industry": None,
    "Description": None,
    "Website URL": None,
    "Headquarters": None,
    "Current_Headcount": None,
    "Percent_Change_in_HC": None,
    "Change_in_HC": None,
    "CEO Name": None,
    "CEO Profile Link": None,
    "Key People": None
}

# 1. NAICS Code
try:
    naics_element = wait.until(EC.presence_of_element_located(
        (By.XPATH, '//div[@id="naics"]//span[@id="naicsLessTextSpan"]/b')))
    company_data["NAICS Code"] = naics_element.text.strip()
except Exception as e:
    print("NAICS Code not found:", e)

# 2. Industry
try:
    industry_element = wait.until(EC.presence_of_element_located(
        (By.CSS_SELECTOR, 'h6[data-testid="company-industry-mini-body"]')))
    company_data["Industry"] = industry_element.text.strip()
except Exception as e:
    print("Industry not found!")

# 3. Description
try:
    view_all_button = wait.until(EC.element_to_be_clickable(
        (By.XPATH, '//a[contains(text(), "VIEW ALL")]')))
    view_all_button.click()
    time.sleep(1)
    description_element = wait.until(EC.presence_of_element_located((By.ID, "compDesc")))
    description = description_element.text.strip()
    if "VIEW LESS" in description:
        description = description.split("VIEW LESS")[0].strip()
    company_data["Description"] = description
except Exception as e:
    print("Description not found!")

# 4. Website URL
try:
    cpweblink_element = wait.until(EC.presence_of_element_located((By.CLASS_NAME, "cpweblink")))
    company_data["Website URL"] = cpweblink_element.find_element(By.TAG_NAME, "a").get_attribute("href")
except Exception as e:
    print("Website URL not found!")

# 5. Headquarters
try:
    block_label_element = wait.until(EC.presence_of_element_located((By.CLASS_NAME, "blockLabel")))
    hq_element = block_label_element.find_element(By.XPATH, "following-sibling::*[1]")
    lines = hq_element.text.splitlines()
    company_data["Headquarters"] = ', '.join([line.strip() for line in lines if line.strip()])
except Exception as e:
    print("Headquarters not found!")

# 6. Current Headcount
try:
    headcount_element = wait.until(EC.presence_of_element_located((
        By.XPATH, '//div[@class="spg-d-flex"]/h6[@class="spg-heading spg-heading--xxsmall spg-text-nowrap"]')))
    company_data["Current_Headcount"] = headcount_element.text.strip()
except Exception as e:
    print("Current Headcount not found!")

# 7. Percent Change in Headcount
try:
    percent_change = wait.until(EC.presence_of_element_located((By.XPATH, '//strong[@class="spg-ml-xs"]')))
    company_data["Percent_Change_in_HC"] = percent_change.text.strip().replace('%', '')
except Exception as e:
    print("Percent Change in Headcount not found!")

# 8. Change in Headcount Direction
try:
    change_type = wait.until(EC.presence_of_element_located((By.XPATH,
        '//div[contains(@class, "spg-d-flex") and contains(@class, "spg-align-center")]//span[@data-icon]')))
    icon_html = change_type.get_attribute('outerHTML')
    if 'caret-up' in icon_html:
        company_data["Change_in_HC"] = 1
    elif 'caret-down' in icon_html:
        company_data["Change_in_HC"] = -1
    else:
        company_data["Change_in_HC"] = 0
except Exception as e:
    print("Change in Headcount direction not found!")

# 9. CEO Name and Profile Link
try:
    ceo_element = wait.until(EC.presence_of_element_located((
        By.CSS_SELECTOR, 'a[data-testid="ribbon-ceo"].spg-text-medium.css-12tcig2.css-1b9i013')))
    company_data["CEO Name"] = ceo_element.text
    company_data["CEO Profile Link"] = ceo_element.get_attribute("href")
except Exception as e:
    print("CEO information not found!")

# 10. Officers
# Click the "More" link under Officers & Directors
more_button = wait.until(EC.element_to_be_clickable((By.XPATH, '//div[@id="offDirectors"]//a[contains(text(), "More")]')))
driver.execute_script("arguments[0].click();", more_button)
logging.info("Clicked on 'More' under Officers & Directors.")
time.sleep(5)

# Step 2: Wait for the table to appear
try:
    wait.until(EC.presence_of_element_located((By.ID, "section_1_control_23_section_0_control_71_i2_grid_table")))
    table = driver.find_element(By.ID, 'section_1_control_23_section_0_control_71_i2_grid_table')
except Exception as e:
    logging.error(f"Table not found: {e}")
    driver.quit()
    exit()

# Step 3: Loop through rows and print all <td> contents
rows = table.find_elements(By.XPATH, './/tbody/tr')
merged_data = []
for i, row in enumerate(rows):
    try:
        name = row.find_element(By.XPATH, './/td[@data-colkey="_NameFromDoc"]').text.strip()
        title = row.find_element(By.XPATH, './/td[@data-colkey="_Title"]/div').text.strip()
        merged_data.append(f" {title} : {name if name else '[No Name]'}")
    except Exception as e:
        logging.warning(f"Could not process row {i+1}: {e}")
final_output = '; '.join(merged_data)
company_data["Key People"] = final_output

# Optional: Close the browser
# driver.quit()


# Create a DataFrame
df2 = pd.DataFrame([company_data])

# Convert columns to numeric, errors='coerce' will turn invalid parsing into NaN
for col, dtype in {
    'Current_Headcount': 'Int64',
    'Change_in_HC': 'Int64',
    'Percent_Change_in_HC': 'float'}.items():
    df2[col] = pd.to_numeric(df2[col], errors='coerce').astype(dtype)

if (
    df2['Current_Headcount'].isnull().any() or
    df2['Percent_Change_in_HC'].isnull().any() or
    df2['Change_in_HC'].isnull().any()):
    df2['LY_Headcount'] = pd.NA
else:
    df2['LY_Headcount'] = (df2['Current_Headcount'] / (1 + (df2['Change_in_HC']*df2['Percent_Change_in_HC'])/100)).round().astype('Int64')
# Reorder columns to bring LY_Headcount after Change_in_HC
cols = list(df2.columns)
if "LY_Headcount" in cols and "Change_in_HC" in cols:
    ly_idx = cols.index("LY_Headcount")
    chg_idx = cols.index("Change_in_HC")
    # Remove LY_Headcount from its current position
    cols.pop(ly_idx)
    # Insert LY_Headcount after Change_in_HC
    cols.insert(chg_idx + 1, "LY_Headcount")
    df2 = df2[cols]
df2


2025-07-14 16:25:07 - INFO - Opened Capital IQ login page.
2025-07-14 16:25:12 - INFO - Email entered.
2025-07-14 16:25:12 - INFO - Email submitted.
2025-07-14 16:25:13 - INFO - Password submitted.
2025-07-14 16:25:13 - INFO - Login successful and page loaded.
2025-07-14 16:25:22 - INFO - Cookie popup accepted.
2025-07-14 16:25:23 - INFO - Navigated to search results for: Polygon Co
2025-07-14 16:25:33 - INFO - No cookie popup found or already handled.


✅ Match found: Polygon Company, Inc. in Walkerton


2025-07-14 16:25:43 - INFO - Clicked on 'More' under Officers & Directors.


,NAICS Code,Industry,Description,Website URL,Headquarters,Current_Headcount,Percent_Change_in_HC,Change_in_HC,LY_Headcount,CEO Name,CEO Profile Link,Key People
0,326121 - Unlaminated Plastics Profile Shape Ma...,Commodity Chemicals,"Polygon Company, Inc., an engineered materials...",http://www.polygoncomposites.com/,"103 Industrial Park Drive, PO Box 176, Walkert...",71,8.0,-1,77,Bree Katulak,https://www.capitaliq.spglobal.com/web/client#...,President & CEO : Bree Katulak; Vice Preside...


In [ ]:
# for debugging and printing HTML structure
# # Wait for JS-rendered content
# time.sleep(5)

# # Get the full HTML after JS execution
# html = driver.execute_script("return document.body.innerHTML;")

# # Save it to a file for inspection
# with open("capitaliq_snapshot.html", "w", encoding="utf-8") as f:
#     f.write(html)

# print("✅ Saved page snapshot to 'capitaliq_snapshot.html'")

#### Trial - Input of multiple companies and cities

In [2]:
import os
import time
import logging
import traceback
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains

# Setup logging
logging.basicConfig(
    level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
    force=True)  # Force logging to overwrite any previous settings

# Load credentials from environment variables (recommended)
EMAIL = "bmorris@lacydiversified.com"
PASSWORD = "Oiaglobal25!"

employer_city_list = [{"Employer Name": "#1 Asian Buffet", "City": "Linton"},
                      {"Employer Name": "#1 Dealer Wholesale", "City": "Indianapolis"},
                      {"Employer Name": "#1 Strategic Solutions Llc", "City": "Indianapolis"},
                      {"Employer Name": "#thatolife Llp", "City": "Indianapolis"}]


# Start browser
driver = webdriver.Chrome()
wait = WebDriverWait(driver, 20)
results = []

# Step 1: Login
driver.get("https://www.capitaliq.spglobal.com")
email_input = wait.until(EC.presence_of_element_located((By.ID, "input28")))
email_input.clear()
email_input.send_keys(EMAIL)
wait.until(EC.element_to_be_clickable((By.XPATH, '//input[@type="submit" and @value="Next"]'))).click()
password_input = wait.until(EC.presence_of_element_located((By.XPATH, '//input[@placeholder="Password"]')))
password_input.send_keys(PASSWORD)
wait.until(EC.element_to_be_clickable((By.XPATH, '//input[@type="submit" and @value="Sign In"]'))).click()
wait.until(EC.presence_of_element_located((By.XPATH, '//div[contains(@class, "header")]')))

# Step 2: Accept cookie popup if present
try:
    cookie_button = WebDriverWait(driver, 5).until(
        EC.element_to_be_clickable((By.XPATH, '//button[contains(text(), "Accept") or contains(text(), "Agree")]')))
    cookie_button.click()
except:
    pass

# Step 3: Loop through employers
for entry in employer_city_list:
    search_term = entry["Employer Name"]
    search_city = entry["City"]
    search_url = f"https://www.capitaliq.spglobal.com/apisv3/spg-webplatform-core/search/searchResults?q={search_term.replace(' ', '%20')}&vertical=private_companies_mi-gss"
    driver.get(search_url)
    wait.until(EC.presence_of_element_located((By.CLASS_NAME, "css-7mog2u")))

    # Change results per page to 50
    try:
        show_label = wait.until(EC.presence_of_element_located((By.CLASS_NAME, "css-7mog2u")))
        driver.execute_script("arguments[0].scrollIntoView(true);", show_label)
        time.sleep(1)
        dropdown_trigger = wait.until(EC.element_to_be_clickable((By.XPATH, '//button[@type="button" and @title="20"]')))
        driver.execute_script("arguments[0].click();", dropdown_trigger)
        time.sleep(1)
        options = wait.until(EC.presence_of_all_elements_located((By.XPATH, '//div[@title="50" and contains(@class, "css-11d71qm")]')))
        for option in options:
            if option.is_displayed():
                driver.execute_script("arguments[0].click();", option)
                time.sleep(3)
                break
    except Exception as e:
        logging.warning(f"Dropdown error: {e}")

    # Search for match
    match_found = False
    try:
        WebDriverWait(driver, 10).until(
            EC.presence_of_all_elements_located((By.XPATH, '//a[contains(@href, "/web/client#company/profile?id=")]'))
        )
        company_links = driver.find_elements(By.XPATH, '//a[contains(@href, "/web/client#company/profile?id=")]')
        for link in company_links:
            try:
                container = link.find_element(By.XPATH, './ancestor::div[3]')
                if search_city.lower() in container.text.lower():
                    results.append({"Employer Name": search_term, "City": search_city, "Match Found": True})
                    match_found = True
                    break
            except:
                continue
        if not match_found:
            results.append({"Employer Name": search_term, "City": search_city, "Match Found": False})
    except Exception as e:
        logging.error(f"Search error for {search_term}: {e}")
        results.append({"Employer Name": search_term, "City": search_city, "Match Found": False})

# Wrap up
driver.quit()
df = pd.DataFrame(results)
df

TimeoutException: Message: 
Stacktrace:
	GetHandleVerifier [0x0x7ff736236f75+76917]
	GetHandleVerifier [0x0x7ff736236fd0+77008]
	(No symbol) [0x0x7ff735fe9dea]
	(No symbol) [0x0x7ff736040256]
	(No symbol) [0x0x7ff73604050c]
	(No symbol) [0x0x7ff736093887]
	(No symbol) [0x0x7ff7360684af]
	(No symbol) [0x0x7ff73609065c]
	(No symbol) [0x0x7ff736068243]
	(No symbol) [0x0x7ff736031431]
	(No symbol) [0x0x7ff7360321c3]
	GetHandleVerifier [0x0x7ff73650d2ad+3051437]
	GetHandleVerifier [0x0x7ff736507903+3028483]
	GetHandleVerifier [0x0x7ff73652589d+3151261]
	GetHandleVerifier [0x0x7ff73625183e+185662]
	GetHandleVerifier [0x0x7ff7362596ff+218111]
	GetHandleVerifier [0x0x7ff73623faf4+112628]
	GetHandleVerifier [0x0x7ff73623fca9+113065]
	GetHandleVerifier [0x0x7ff736226c78+10616]
	BaseThreadInitThunk [0x0x7ffab50fe8d7+23]
	RtlUserThreadStart [0x0x7ffab727c34c+44]


#### trial for LinkedIN Professionals

In [ ]:
import os
import time
import logging
import traceback
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains


# Setup logging
logging.basicConfig(
    level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
    force=True)  # Force logging to overwrite any previous settings

# Load credentials from environment variables (recommended)
EMAIL = "bmorris@lacydiversified.com"
PASSWORD = "Oiaglobal25!"

# Setup Chrome options
options = Options()
# options.add_argument("--headless")  # Uncomment if running in headless mode
driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 20)

# Step 1: Open Capital IQ login page
driver.get("https://www.capitaliq.spglobal.com")
logging.info("Opened Capital IQ login page.")

# Step 2: Enter email
email_input = wait.until(EC.presence_of_element_located((By.ID, "input28")))
email_input.clear()
email_input.send_keys(EMAIL)
logging.info("Email entered.")

# Step 3: Click "Next"
next_button = wait.until(EC.element_to_be_clickable((By.XPATH, '//input[@type="submit" and @value="Next"]')))
next_button.click()
logging.info("Email submitted.")

# Step 4: Enter password
password_input = wait.until(EC.presence_of_element_located((By.XPATH, '//input[@placeholder="Password"]')))
password_input.send_keys(PASSWORD)

# Step 5: Click "Sign In"
sign_in_button = wait.until(EC.element_to_be_clickable((By.XPATH, '//input[@type="submit" and @value="Sign In"]')))
sign_in_button.click()
logging.info("Password submitted.")

# Step 6: Wait for login to complete
wait.until(EC.presence_of_element_located((By.XPATH, '//div[contains(@class, "header")]')))
logging.info("Login successful and page loaded.")

# Step 7: Accept cookie popup if present
try:
    cookie_button = WebDriverWait(driver, 10).until(
        EC.element_to_be_clickable((By.XPATH, '//button[contains(text(), "Accept") or contains(text(), "Agree")]')))
    cookie_button.click()
    logging.info("Cookie popup accepted.")
except:
    logging.info("No cookie popup found or already handled.")

# Step 8: Navigate directly to the search results URL
search_term = "Polygon Co"
search_city = "Walkerton"
search_url = f"https://www.capitaliq.spglobal.com/apisv3/spg-webplatform-core/search/searchResults?q={search_term.replace(' ', '%20')}&vertical=private_companies_mi-gss"
driver.get(search_url)
logging.info(f"Navigated to search results for: {search_term}")
wait.until(EC.presence_of_element_located((By.CLASS_NAME, "css-7mog2u")))


# Step 8.1: Handle cookie popup if present
try:
    cookie_button = WebDriverWait(driver, 10).until(
        EC.element_to_be_clickable((By.XPATH, '//button[contains(text(), "Accept") or contains(text(), "Agree")]')))
    cookie_button.click()
    logging.info("Cookie popup accepted.")
except:
    logging.info("No cookie popup found or already handled.")

# Step 9: Change results per page to 50
try:
    show_label = wait.until(EC.presence_of_element_located((By.CLASS_NAME, "css-7mog2u")))
    driver.execute_script("arguments[0].scrollIntoView(true);", show_label)
    time.sleep(1)

    dropdown_trigger = wait.until(EC.element_to_be_clickable((By.XPATH, '//button[@type="button" and @title="20"]')))
    driver.execute_script("arguments[0].click();", dropdown_trigger)
    time.sleep(1)

    options = wait.until(EC.presence_of_all_elements_located((By.XPATH, '//div[@title="50" and contains(@class, "css-11d71qm")]')))
    for option in options:
        if option.is_displayed():
            driver.execute_script("arguments[0].click();", option)
            print("✅ Changed results per page to 50")
            time.sleep(3)
            break
except Exception as e:
    logging.error(f"Error while changing results per page: {e}")

# Step 10: Wait for company cards to load and click matching company
try:
    WebDriverWait(driver, 10).until(
        EC.presence_of_all_elements_located((By.XPATH, '//a[contains(@href, "/web/client#company/profile?id=")]'))
    )

    company_links = driver.find_elements(By.XPATH, '//a[contains(@href, "/web/client#company/profile?id=")]')
    match_found = False

    for link in company_links:
        try:
            # Go up to the grandparent container
            container = link.find_element(By.XPATH, './ancestor::div[3]')
            container_text = container.text

            if search_city.lower() in container_text.lower():
                print(f"✅ Match found: {link.text.strip()} in {search_city}")
                driver.execute_script("arguments[0].click();", link)
                match_found = True
                break

        except Exception as e:
            print(f"⚠️ Error processing link: {e}")

    if not match_found:
        print(f"❗ No matching company found for city: {search_city}")

except Exception as e:
    print(f"❌ Error locating company links: {e}")

# Initialize a dictionary to hold the extracted data
company_data = {
    "NAICS Code": None,
    "Industry": None,
    "Description": None,
    "Website URL": None,
    "Headquarters": None,
    "Current_Headcount": None,
    "Percent_Change_in_HC": None,
    "Change_in_HC": None,
    "CEO Name": None,
    "CEO Profile Link": None,
    "LinkedIN URL": None,
    "Key People": None
}

# 1. NAICS Code
try:
    naics_element = wait.until(EC.presence_of_element_located(
        (By.XPATH, '//div[@id="naics"]//span[@id="naicsLessTextSpan"]/b')))
    company_data["NAICS Code"] = naics_element.text.strip()
except Exception as e:
    print("NAICS Code not found:", e)

# 2. Industry
try:
    industry_element = wait.until(EC.presence_of_element_located(
        (By.CSS_SELECTOR, 'h6[data-testid="company-industry-mini-body"]')))
    company_data["Industry"] = industry_element.text.strip()
except Exception as e:
    print("Industry not found!")

# 3. Description
try:
    view_all_button = wait.until(EC.element_to_be_clickable(
        (By.XPATH, '//a[contains(text(), "VIEW ALL")]')))
    view_all_button.click()
    time.sleep(1)
    description_element = wait.until(EC.presence_of_element_located((By.ID, "compDesc")))
    description = description_element.text.strip()
    if "VIEW LESS" in description:
        description = description.split("VIEW LESS")[0].strip()
    company_data["Description"] = description
except Exception as e:
    print("Description not found!")

# 4. Website URL
try:
    cpweblink_element = wait.until(EC.presence_of_element_located((By.CLASS_NAME, "cpweblink")))
    company_data["Website URL"] = cpweblink_element.find_element(By.TAG_NAME, "a").get_attribute("href")
except Exception as e:
    print("Website URL not found!")

# 5. Headquarters
try:
    block_label_element = wait.until(EC.presence_of_element_located((By.CLASS_NAME, "blockLabel")))
    hq_element = block_label_element.find_element(By.XPATH, "following-sibling::*[1]")
    lines = hq_element.text.splitlines()
    company_data["Headquarters"] = ', '.join([line.strip() for line in lines if line.strip()])
except Exception as e:
    print("Headquarters not found!")

# 6. Current Headcount
try:
    headcount_element = wait.until(EC.presence_of_element_located((
        By.XPATH, '//div[@class="spg-d-flex"]/h6[@class="spg-heading spg-heading--xxsmall spg-text-nowrap"]')))
    company_data["Current_Headcount"] = headcount_element.text.strip()
except Exception as e:
    print("Current Headcount not found!")

# 7. Percent Change in Headcount
try:
    percent_change = wait.until(EC.presence_of_element_located((By.XPATH, '//strong[@class="spg-ml-xs"]')))
    company_data["Percent_Change_in_HC"] = percent_change.text.strip().replace('%', '')
except Exception as e:
    print("Percent Change in Headcount not found!")

# 8. Change in Headcount Direction
try:
    change_type = wait.until(EC.presence_of_element_located((By.XPATH,
        '//div[contains(@class, "spg-d-flex") and contains(@class, "spg-align-center")]//span[@data-icon]')))
    icon_html = change_type.get_attribute('outerHTML')
    if 'caret-up' in icon_html:
        company_data["Change_in_HC"] = 1
    elif 'caret-down' in icon_html:
        company_data["Change_in_HC"] = -1
    else:
        company_data["Change_in_HC"] = 0
except Exception as e:
    print("Change in Headcount direction not found!")

# 9. CEO Name and Profile Link
try:
    ceo_element = wait.until(EC.presence_of_element_located((
        By.CSS_SELECTOR, 'a[data-testid="ribbon-ceo"].spg-text-medium.css-12tcig2.css-1b9i013')))
    company_data["CEO Name"] = ceo_element.text
    company_data["CEO Profile Link"] = ceo_element.get_attribute("href")
except Exception as e:
    print("CEO information not found!")

# 10: Get LinkedIN URL of Company
try:
    wait = WebDriverWait(driver, 10)
    
    # Find the span with the text and get its parent anchor tag
    span = wait.until(EC.presence_of_element_located(
        (By.XPATH, "//span[contains(text(), 'View LinkedIn Professionals')]")))
    company_data["LinkedIN URL"] = span.find_element(By.XPATH, "./ancestor::a").get_attribute("href")

except Exception as e:
    print("LinkedIn link not found:", e)

# 11. Officers
# Step 1: Click the "More" link under Officers & Directors
more_button = wait.until(EC.element_to_be_clickable((By.XPATH, '//div[@id="offDirectors"]//a[contains(text(), "More")]')))
driver.execute_script("arguments[0].click();", more_button)
logging.info("Clicked on 'More' under Officers & Directors.")
time.sleep(5)

# Step 2: Wait for the table to appear
# Wait for the AG Grid wrapper to appear
wait.until(EC.presence_of_element_located((By.XPATH, "//div[contains(@class, 'ag-root-wrapper')]")))
# Locate the grid rows
rows = driver.find_elements(By.XPATH, "//div[contains(@class, 'ag-center-cols-container')]/div[contains(@class, 'ag-row')]")

merged_data = []
for i, row in enumerate(rows):
    try:
        # Each cell is a div with class ag-cell
        cells = row.find_elements(By.XPATH, ".//div[contains(@class, 'ag-cell')]")
        if len(cells) >= 2:
            name = cells[0].text.strip()
            title = cells[1].text.strip()
            merged_data.append(f"{title} : {name if name else '[No Name]'}")
    except Exception as e:
        logging.warning(f"Could not process row {i+1}: {e}")

final_output = '; '.join(merged_data)
company_data["Key People"] = final_output
# Navigate to the company's LinkedIn "people" page


# Reorder columns to bring LY_Headcount after Change_in_HC
cols = list(df2.columns)
if "LY_Headcount" in cols and "Change_in_HC" in cols:
    ly_idx = cols.index("LY_Headcount")
    chg_idx = cols.index("Change_in_HC")
    # Remove LY_Headcount from its current position
    cols.pop(ly_idx)
    # Insert LY_Headcount after Change_in_HC
    cols.insert(chg_idx + 1, "LY_Headcount")
    df2 = df2[cols]
df2

2025-07-23 11:24:13 - INFO - Opened Capital IQ login page.
2025-07-23 11:24:21 - INFO - Email entered.
2025-07-23 11:24:21 - INFO - Email submitted.
2025-07-23 11:24:23 - INFO - Password submitted.
2025-07-23 11:24:23 - INFO - Login successful and page loaded.
2025-07-23 11:24:37 - INFO - Cookie popup accepted.
2025-07-23 11:24:38 - INFO - Navigated to search results for: Polygon Co
2025-07-23 11:24:49 - INFO - No cookie popup found or already handled.


✅ Changed results per page to 50
✅ Match found: Polygon Company, Inc. in Walkerton


2025-07-23 11:25:03 - INFO - Clicked on 'More' under Officers & Directors.


NoSuchElementException: Message: no such element: Unable to locate element: {"method":"css selector","selector":".hui-link"}
  (Session info: chrome=138.0.7204.158); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#no-such-element-exception
Stacktrace:
	GetHandleVerifier [0x0x7ff600c1e935+77845]
	GetHandleVerifier [0x0x7ff600c1e990+77936]
	(No symbol) [0x0x7ff6009d9cda]
	(No symbol) [0x0x7ff600a306aa]
	(No symbol) [0x0x7ff600a3095c]
	(No symbol) [0x0x7ff600a83d07]
	(No symbol) [0x0x7ff600a5890f]
	(No symbol) [0x0x7ff600a80b07]
	(No symbol) [0x0x7ff600a586a3]
	(No symbol) [0x0x7ff600a21791]
	(No symbol) [0x0x7ff600a22523]
	GetHandleVerifier [0x0x7ff600ef684d+3059501]
	GetHandleVerifier [0x0x7ff600ef0c0d+3035885]
	GetHandleVerifier [0x0x7ff600f10400+3164896]
	GetHandleVerifier [0x0x7ff600c38c3e+185118]
	GetHandleVerifier [0x0x7ff600c4054f+216111]
	GetHandleVerifier [0x0x7ff600c272e4+113092]
	GetHandleVerifier [0x0x7ff6

In [8]:
company_data

{'NAICS Code': '326121 - Unlaminated Plastics Profile Shape Manufacturing',
 'Industry': 'Commodity Chemicals',
 'Description': 'Polygon Company, Inc., an engineered materials company, manufactures and supplies composite bearing, non-gelcoat composite pneumatic cylinders, composite laparoscopic surgical tubing, and dielectric motor insulation worldwide. It offers rigid medical tubing for medical applications; self-lubricating composite bearings and bushings for man-lift and construction machinery equipment industry; pneumatic and hydraulic cylinders for fluid power applications; and dielectric tubing for electrical insulation applications. It serves Fortune 500 companies primarily operating in construction equipment, oil drilling, laparoscopic surgical tubing, and agriculture equipment. Polygon Company, Inc. was founded in 1949 and is based in Walkerton, Indiana with facilities in North America; and Xiamen, China.',
 'Website URL': 'http://www.polygoncomposites.com/',
 'Headquarters': 

#### extra

In [ ]:
# Initialize a dictionary to hold the extracted data
company_data = {
    "NAICS Code": None,
    "Industry": None,
    "Description": None,
    "Website URL": None,
    "Headquarters": None,
    "Current_Headcount": None,
    "Percent_Change_in_HC": None,
    "Change_in_HC": None,
    "CEO Name": None,
    "CEO Profile Link": None,
    "Key People": None
}

# 1. NAICS Code
try:
    naics_element = wait.until(EC.presence_of_element_located(
        (By.XPATH, '//div[@id="naics"]//span[@id="naicsLessTextSpan"]/b')))
    company_data["NAICS Code"] = naics_element.text.strip()
except Exception as e:
    print("NAICS Code not found:", e)

# 2. Industry
try:
    industry_element = wait.until(EC.presence_of_element_located(
        (By.CSS_SELECTOR, 'h6[data-testid="company-industry-mini-body"]')))
    company_data["Industry"] = industry_element.text.strip()
except Exception as e:
    print("Industry not found!")

# 3. Description
try:
    view_all_button = wait.until(EC.element_to_be_clickable(
        (By.XPATH, '//a[contains(text(), "VIEW ALL")]')))
    view_all_button.click()
    time.sleep(1)
    description_element = wait.until(EC.presence_of_element_located((By.ID, "compDesc")))
    description = description_element.text.strip()
    if "VIEW LESS" in description:
        description = description.split("VIEW LESS")[0].strip()
    company_data["Description"] = description
except Exception as e:
    print("Description not found!")

# 4. Website URL
try:
    cpweblink_element = wait.until(EC.presence_of_element_located((By.CLASS_NAME, "cpweblink")))
    company_data["Website URL"] = cpweblink_element.find_element(By.TAG_NAME, "a").get_attribute("href")
except Exception as e:
    print("Website URL not found!")

# 5. Headquarters
try:
    block_label_element = wait.until(EC.presence_of_element_located((By.CLASS_NAME, "blockLabel")))
    hq_element = block_label_element.find_element(By.XPATH, "following-sibling::*[1]")
    lines = hq_element.text.splitlines()
    company_data["Headquarters"] = ', '.join([line.strip() for line in lines if line.strip()])
except Exception as e:
    print("Headquarters not found!")

# 6. Current Headcount
try:
    headcount_element = wait.until(EC.presence_of_element_located((
        By.XPATH, '//div[@class="spg-d-flex"]/h6[@class="spg-heading spg-heading--xxsmall spg-text-nowrap"]')))
    company_data["Current_Headcount"] = headcount_element.text.strip()
except Exception as e:
    print("Current Headcount not found!")

# 7. Percent Change in Headcount
try:
    percent_change = wait.until(EC.presence_of_element_located((By.XPATH, '//strong[@class="spg-ml-xs"]')))
    company_data["Percent_Change_in_HC"] = percent_change.text.strip().replace('%', '')
except Exception as e:
    print("Percent Change in Headcount not found!")

# 8. Change in Headcount Direction
try:
    change_type = wait.until(EC.presence_of_element_located((By.XPATH,
        '//div[contains(@class, "spg-d-flex") and contains(@class, "spg-align-center")]//span[@data-icon]')))
    icon_html = change_type.get_attribute('outerHTML')
    if 'caret-up' in icon_html:
        company_data["Change_in_HC"] = 1
    elif 'caret-down' in icon_html:
        company_data["Change_in_HC"] = -1
    else:
        company_data["Change_in_HC"] = 0
except Exception as e:
    print("Change in Headcount direction not found!")

# 9. CEO Name and Profile Link
try:
    ceo_element = wait.until(EC.presence_of_element_located((
        By.CSS_SELECTOR, 'a[data-testid="ribbon-ceo"].spg-text-medium.css-12tcig2.css-1b9i013')))
    company_data["CEO Name"] = ceo_element.text
    company_data["CEO Profile Link"] = ceo_element.get_attribute("href")
except Exception as e:
    print("CEO information not found!")

# 10. Officers
# Click the "More" link under Officers & Directors
more_button = wait.until(EC.element_to_be_clickable((By.XPATH, '//div[@id="offDirectors"]//a[contains(text(), "More")]')))
driver.execute_script("arguments[0].click();", more_button)
logging.info("Clicked on 'More' under Officers & Directors.")
time.sleep(5)

# Step 2: Wait for the table to appear
try:
    wait.until(EC.presence_of_element_located((By.ID, "section_1_control_23_section_0_control_71_i2_grid_table")))
    table = driver.find_element(By.ID, 'section_1_control_23_section_0_control_71_i2_grid_table')
except Exception as e:
    logging.error(f"Table not found: {e}")
    driver.quit()
    exit()

# Step 3: Loop through rows and print all <td> contents
rows = table.find_elements(By.XPATH, './/tbody/tr')
merged_data = []
for i, row in enumerate(rows):
    try:
        name = row.find_element(By.XPATH, './/td[@data-colkey="_NameFromDoc"]').text.strip()
        title = row.find_element(By.XPATH, './/td[@data-colkey="_Title"]/div').text.strip()
        merged_data.append(f" {title} : {name if name else '[No Name]'}")
    except Exception as e:
        logging.warning(f"Could not process row {i+1}: {e}")
final_output = '; '.join(merged_data)
company_data["Key People"] = final_output

# Optional: Close the browser
# driver.quit()


# Create a DataFrame
df2 = pd.DataFrame([company_data])

# Convert columns to numeric, errors='coerce' will turn invalid parsing into NaN
for col, dtype in {
    'Current_Headcount': 'Int64',
    'Change_in_HC': 'Int64',
    'Percent_Change_in_HC': 'float'}.items():
    df2[col] = pd.to_numeric(df2[col], errors='coerce').astype(dtype)

if (
    df2['Current_Headcount'].isnull().any() or
    df2['Percent_Change_in_HC'].isnull().any() or
    df2['Change_in_HC'].isnull().any()):
    df2['LY_Headcount'] = pd.NA
else:
    df2['LY_Headcount'] = (df2['Current_Headcount'] / (1 + (df2['Change_in_HC']*df2['Percent_Change_in_HC'])/100)).round().astype('Int64')
# Reorder columns to bring LY_Headcount after Change_in_HC
cols = list(df2.columns)
if "LY_Headcount" in cols and "Change_in_HC" in cols:
    ly_idx = cols.index("LY_Headcount")
    chg_idx = cols.index("Change_in_HC")
    # Remove LY_Headcount from its current position
    cols.pop(ly_idx)
    # Insert LY_Headcount after Change_in_HC
    cols.insert(chg_idx + 1, "LY_Headcount")
    df2 = df2[cols]
df2

In [ ]:
# 1. NAICS Code
try:
    naics_element = wait.until(EC.presence_of_element_located(
        (By.XPATH, '//div[@id="naics"]//span[@id="naicsLessTextSpan"]/b')))
    company_data["NAICS Code"] = naics_element.text.strip()
except Exception as e:
    print("NAICS Code not found:", e)

# 2. Industry
try:
    industry_element = wait.until(EC.presence_of_element_located(
        (By.CSS_SELECTOR, 'h6[data-testid="company-industry-mini-body"]')))
    company_data["Industry"] = industry_element.text.strip()
except Exception as e:
    print("Industry not found!")

# 3. Description
try:
    view_all_button = wait.until(EC.element_to_be_clickable(
        (By.XPATH, '//a[contains(text(), "VIEW ALL")]')))
    view_all_button.click()
    time.sleep(1)
    description_element = wait.until(EC.presence_of_element_located((By.ID, "compDesc")))
    description = description_element.text.strip()
    if "VIEW LESS" in description:
        description = description.split("VIEW LESS")[0].strip()
    company_data["Description"] = description
except Exception as e:
    print("Description not found!")

# 4. Website URL
try:
    cpweblink_element = wait.until(EC.presence_of_element_located((By.CLASS_NAME, "cpweblink")))
    company_data["Website URL"] = cpweblink_element.find_element(By.TAG_NAME, "a").get_attribute("href")
except Exception as e:
    print("Website URL not found!")

# 5. Headquarters
try:
    block_label_element = wait.until(EC.presence_of_element_located((By.CLASS_NAME, "blockLabel")))
    hq_element = block_label_element.find_element(By.XPATH, "following-sibling::*[1]")
    lines = hq_element.text.splitlines()
    company_data["Headquarters"] = ', '.join([line.strip() for line in lines if line.strip()])
except Exception as e:
    print("Headquarters not found!")

# 6. Current Headcount
try:
    headcount_element = wait.until(EC.presence_of_element_located((
        By.XPATH, '//div[@class="spg-d-flex"]/h6[@class="spg-heading spg-heading--xxsmall spg-text-nowrap"]')))
    company_data["Current_Headcount"] = headcount_element.text.strip()
except Exception as e:
    print("Current Headcount not found!")

# 7. Percent Change in Headcount
try:
    percent_change = wait.until(EC.presence_of_element_located((By.XPATH, '//strong[@class="spg-ml-xs"]')))
    company_data["Percent_Change_in_HC"] = percent_change.text.strip().replace('%', '')
except Exception as e:
    print("Percent Change in Headcount not found!")

# 8. Change in Headcount Direction
try:
    change_type = wait.until(EC.presence_of_element_located((By.XPATH,
        '//div[contains(@class, "spg-d-flex") and contains(@class, "spg-align-center")]//span[@data-icon]')))
    icon_html = change_type.get_attribute('outerHTML')
    if 'caret-up' in icon_html:
        company_data["Change_in_HC"] = 1
    elif 'caret-down' in icon_html:
        company_data["Change_in_HC"] = -1
    else:
        company_data["Change_in_HC"] = 0
except Exception as e:
    print("Change in Headcount direction not found!")

# 9. CEO Name and Profile Link
try:
    ceo_element = wait.until(EC.presence_of_element_located((
        By.CSS_SELECTOR, 'a[data-testid="ribbon-ceo"].spg-text-medium.css-12tcig2.css-1b9i013')))
    company_data["CEO Name"] = ceo_element.text
    company_data["CEO Profile Link"] = ceo_element.get_attribute("href")
except Exception as e:
    print("CEO information not found!")

# 10. Officers
# Click the "More" link under Officers & Directors
more_button = wait.until(EC.element_to_be_clickable((By.XPATH, '//div[@id="offDirectors"]//a[contains(text(), "More")]')))
driver.execute_script("arguments[0].click();", more_button)
logging.info("Clicked on 'More' under Officers & Directors.")
time.sleep(5)

# Step 2: Wait for the table to appear
try:
    wait.until(EC.presence_of_element_located((By.ID, "section_1_control_23_section_0_control_71_i2_grid_table")))
    table = driver.find_element(By.ID, 'section_1_control_23_section_0_control_71_i2_grid_table')
except Exception as e:
    logging.error(f"Table not found: {e}")
    driver.quit()
    exit()

# Step 3: Loop through rows and print all <td> contents
rows = table.find_elements(By.XPATH, './/tbody/tr')
merged_data = []
for i, row in enumerate(rows):
    try:
        name = row.find_element(By.XPATH, './/td[@data-colkey="_NameFromDoc"]').text.strip()
        title = row.find_element(By.XPATH, './/td[@data-colkey="_Title"]/div').text.strip()
        merged_data.append(f" {title} : {name if name else '[No Name]'}")
    except Exception as e:
        logging.warning(f"Could not process row {i+1}: {e}")
final_output = '; '.join(merged_data)
company_data["Key People"] = final_output

